In [1]:
import os


In [2]:
os.environ["CUDA_VISIBLE_DEVICES"] = "6"

In [3]:
import pandas as pd

In [4]:
df_path = "/scratch/yag1/ego4d_data/v2/updated_queries_with_talk.csv"

query_df = pd.read_csv(df_path)

In [5]:
from transformers import ClapModel, ClapProcessor


In [6]:
clap_model = ClapModel.from_pretrained("laion/clap-htsat-fused").to(0)
clap_processor = ClapProcessor.from_pretrained("laion/clap-htsat-fused")


Loading weights:   0%|          | 0/477 [00:00<?, ?it/s]

In [7]:
import torch

In [8]:
def get_clap_text_embedding(text, clip_id, row_num):
    inputs = clap_processor(text=text, return_tensors="pt").to(0)
    with torch.no_grad():
        model_output = clap_model.get_text_features(**inputs)
    
    root_path = "/scratch/yag1/ego4d_data/v2/clips"
    clap_text_embedding_dir = root_path + "/" + clip_id + "/text_embeddings" + "/clap_text_embeddings"
    os.makedirs(clap_text_embedding_dir, exist_ok=True)
    torch.save(model_output.pooler_output[0].cpu(), clap_text_embedding_dir + f"/{row_num}.pt")
    
    return clap_text_embedding_dir + f"/{row_num}.pt"

In [9]:
# query_df["clap_text_embedding"] = query_df["query"].apply(get_clap_text_embedding)

query_df["clap_text_embedding_path"] = query_df.apply(
    lambda row: get_clap_text_embedding(row["clap_query"], row["clip_id"], row.name),
    axis=1)

In [10]:
# !pip install git+https://github.com/google-deepmind/videoprism
# !pip install jax jaxlib
# !pip install --upgrade "jax[cuda12]"

In [11]:
# @title Load model
import jax
import jax.numpy as jnp
from videoprism import models as vp
import numpy as np

In [12]:
print(jax.devices())

[CudaDevice(id=0)]


In [13]:
MODEL_NAME = 'videoprism_lvt_public_v1_base'  # @param ['videoprism_lvt_public_v1_base', 'videoprism_lvt_public_v1_large'] {allow-input: false}
USE_BFLOAT16 = False  # @param { type: "boolean" }

fprop_dtype = jnp.bfloat16 if USE_BFLOAT16 else None
flax_model = vp.get_model(MODEL_NAME, fprop_dtype=fprop_dtype)
loaded_state = vp.load_pretrained_weights(MODEL_NAME)
text_tokenizer = vp.load_text_tokenizer('c4_en')


@jax.jit
def forward_fn(inputs, text_token_ids, text_paddings, train=False):
  return flax_model.apply(
      loaded_state,
      inputs,
      text_token_ids,
      text_paddings,
      train=train,
  )

W0000 00:00:1782066960.102107 1047834 google_auth_provider.cc:196] All attempts to get a Google authentication bearer token failed, returning an empty token. Retrieving token from files failed with "NOT_FOUND: Could not locate the credentials file.". Retrieving token from GCE failed with "FAILED_PRECONDITION: Error executing an HTTP request: libcurl code 6 meaning 'Could not resolve hostname', error details: Could not resolve host: metadata.google.internal".


In [14]:
def get_video_prism_text_embedding(text, clip_id, row_num):
    text_ids, text_paddings = vp.tokenize_texts(text_tokenizer, [text])
    if USE_BFLOAT16:
        text_paddings = text_paddings.astype(jnp.bfloat16)
    _, text_embedding, _ = forward_fn(None, text_ids, text_paddings)
    jax_arr = jax.device_get(text_embedding[0])
    torch_tensor = torch.tensor(np.asarray(jax_arr))
    
    root_path = "/scratch/yag1/ego4d_data/v2/clips"
    video_prism_text_embedding_dir = root_path + "/" + clip_id + "/text_embeddings" + "/video_prism_text_embeddings"
    os.makedirs(video_prism_text_embedding_dir, exist_ok=True)
    torch.save(torch_tensor, video_prism_text_embedding_dir + f"/{row_num}.pt")
    return video_prism_text_embedding_dir + f"/{row_num}.pt"

In [15]:
query_df["video_prism_text_embedding_path"] = query_df.apply(
    lambda row: get_video_prism_text_embedding(row["videoprism_query"], row["clip_id"], row.name),
    axis=1)

2026-06-21 13:36:11.868224: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2026-06-21 13:36:11.868262: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.


In [16]:
from sentence_transformers import SentenceTransformer


In [17]:
# Load the model
model = SentenceTransformer("Qwen/Qwen3-VL-Embedding-2B")


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

Default prompt name is set to 'default'. This prompt will be applied to all inference calls, except if a `prompt` or `prompt_name` parameter is provided.


In [18]:
def get_qwen_text_embedding(text, clip_id, row_num):
    text_embedding = model.encode([text])
    text_embedding = text_embedding.squeeze(0)  # Remove the batch dimension
    torch_tensor = torch.from_numpy(text_embedding)

    root_path = "/scratch/yag1/ego4d_data/v2/clips"
    qwen_text_embedding_dir = root_path + "/" + clip_id + "/text_embeddings" + "/qwen_text_embeddings"
    os.makedirs(qwen_text_embedding_dir, exist_ok=True)
    torch.save(torch_tensor, qwen_text_embedding_dir + f"/{row_num}.pt")
    
    return qwen_text_embedding_dir + f"/{row_num}.pt"

In [19]:
query_df["qwen_text_embedding_path"] = query_df.apply(
    lambda row: get_qwen_text_embedding(row["videoprism_query"], row["clip_id"], row.name),
    axis=1)

In [20]:
query_df.to_csv("/scratch/yag1/ego4d_data/v2/queries_with_talk_and_paths.csv", index=False)